<p>
<font size='5' face='Georgia, Arial'>IIC2233 Apunte Programación Avanzada</font><br>
<font size='1'> Creado el 2026-1 por Equipo Docente IIC2233. </font>
</p>

# Tabla de contenidos

1. [Consultas concurrentes](#Consultas-concurrentes-y-Race-Conditions)
    1. [Enviar múltiples requests en paralelo](#Enviar-múltiples-requests-en-paralelo)
    2. [`ThreadPoolExecutor`: principales métodos](#ThreadPoolExecutor:-principales-métodos)
    3. [Secuencial vs concurrente: comparación de tiempos](#Secuencial-vs-concurrente:-comparación-de-tiempos)

2. [¿Qué es una *Race Condition*?](#¿Qué-es-una-Race-Condition?)

3. [*Race Conditions* desde el cliente](#Race-Conditions-desde-el-cliente)
    1. [Escenario 1: doble submit de formulario](#Escenario-1:-doble-submit-de-formulario)
    2. [Escenario 2: *rate limiting* (código 429)](#Escenario-2:-rate-limiting-(código-429))
    3. [Implementación: backoff con reintentos](#Implementación:-backoff-con-reintentos)

4. [*Race Conditions* en servidores web](#Race-Conditions-en-servidores-web)
    1. [El problema: estado compartido](#El-problema:-estado-compartido)
    2. [Solución: usar un `Lock`](#Solución:-usar-un-Lock)


---

## Consultas concurrentes y *Race Conditions*

### Enviar múltiples requests en paralelo

Hasta ahora hemos enviado las solicitudes **de forma secuencial**: esperamos la respuesta antes de enviar la siguiente. Pero a veces queremos enviar **muchas requests al mismo tiempo** para reducir el tiempo total de espera. Esto se puede hacer con `concurrent.futures.ThreadPoolExecutor`.

```
Secuencial:  [req1]──►[req2]──►[req3]──►  tiempo total = suma de tiempos individuales
Concurrente: [req1]
             [req2]  ←── todas en paralelo, tiempo total ≈ tiempo del más lento
             [req3]
```

### `ThreadPoolExecutor`: principales métodos

`ThreadPoolExecutor` vive en el módulo `concurrent.futures` y gestiona un **pool de hilos** reutilizables. Sus métodos principales son:

| Método | Descripción |
|--------|-------------|
| `ThreadPoolExecutor(max_workers=N)` | Crea el pool con hasta `N` hilos simultáneos. Si se omite, Python elige un valor por defecto basado en los núcleos del sistema. |
| `executor.submit(fn, *args, **kwargs)` | Envía `fn(*args, **kwargs)` al pool y retorna un objeto `Future` **inmediatamente**, sin bloquear. |
| `executor.map(fn, iterable)` | Versión simplificada: aplica `fn` a cada elemento del iterable y retorna los resultados **en orden de entrada**. Bloquea hasta que todos terminen. |
| `future.result()` | Bloquea hasta obtener el valor de retorno del `Future`, o relanza la excepción si la función falló. |
| `as_completed(futures)` | Generador que yield-ea cada `Future` a medida que **termina** (orden de completación, no de envío). |

El patrón recomendado es usar el **context manager** (`with`): al salir del bloque, Python espera a que todos los hilos terminen (`shutdown(wait=True)`) antes de continuar.

```python
with ThreadPoolExecutor(max_workers=4) as executor:
    # submit → devuelve Future sin bloquear
    future = executor.submit(mi_funcion, arg1, arg2)
    resultado = future.result()  # aquí sí bloquea

    # map → más simple, resultados en orden
    resultados = list(executor.map(mi_funcion, lista_de_args))

    # submit + as_completed → resultados en orden de llegada
    futuros = [executor.submit(mi_funcion, x) for x in lista_de_args]
    for f in as_completed(futuros):
        print(f.result())
# al salir del 'with', se espera que todos los hilos terminen
```

In [4]:
import requests
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

BASE = "https://dummyjson.com/"


def obtener_receta(receta_id: int) -> dict:
    """Obtiene una receta individual por su id."""
    respuesta = requests.get(f"{BASE}recipes/{receta_id}")
    return {
        "id": receta_id,
        "status": respuesta.status_code,
        "nombre": respuesta.json().get("name"),
    }


In [5]:
ids_recetas = list(range(1, 11))  # recetas 1 a 10

with ThreadPoolExecutor(max_workers=10) as executor:
    futuros = {executor.submit(obtener_receta, i): i for i in ids_recetas}
    resultados_con = [futuro.result() for futuro in as_completed(futuros)]

for r in sorted(resultados_con, key=lambda x: x["id"]):
    print(f"  Receta {r['id']:2d} [{r['status']}]: {r['nombre']}")


  Receta  1 [200]: Classic Margherita Pizza
  Receta  2 [200]: Vegetarian Stir-Fry
  Receta  3 [200]: Chocolate Chip Cookies
  Receta  4 [200]: Chicken Alfredo Pasta
  Receta  5 [200]: Mango Salsa Chicken
  Receta  6 [200]: Quinoa Salad with Avocado
  Receta  7 [200]: Tomato Basil Bruschetta
  Receta  8 [200]: Beef and Broccoli Stir-Fry
  Receta  9 [200]: Caprese Salad
  Receta 10 [200]: Shrimp Scampi Pasta


### Secuencial vs concurrente: comparación de tiempos

Ahora que conocemos los métodos principales de `ThreadPoolExecutor`, vamos a ver en la práctica la diferencia entre la forma **secuencial** y la forma **concurrente** de enviar requests, comparando el tiempo total de cada enfoque al obtener 10 recetas de la API de _DummyJSON_.


In [6]:

# --- Forma secuencial ---
inicio = time.time()
resultados_sec = [obtener_receta(i) for i in ids_recetas]
tiempo_sec = time.time() - inicio
print(f"Secuencial  → {tiempo_sec:.2f}s  ({len(resultados_sec)} recetas)")

# --- Forma concurrente ---
inicio = time.time()
with ThreadPoolExecutor(max_workers=10) as executor:
    futuros = {executor.submit(obtener_receta, i): i for i in ids_recetas}
    resultados_con = [futuro.result() for futuro in as_completed(futuros)]
tiempo_con = time.time() - inicio
print(f"Concurrente → {tiempo_con:.2f}s  ({len(resultados_con)} recetas)")
print(f"Aceleración : {tiempo_sec / tiempo_con:.1f}x más rápido")


Secuencial  → 3.98s  (10 recetas)
Concurrente → 0.94s  (10 recetas)
Aceleración : 4.3x más rápido


---

## ¿Qué es una *Race Condition*?

Una ***race condition*** (condición de carrera) ocurre cuando **dos o más hilos o procesos acceden y modifican un recurso compartido de forma concurrente**, y el resultado final depende del orden arbitrario en que se ejecutan las operaciones. Como ese orden no está garantizado, el resultado puede ser **incorrecto e impredecible**.

```
Hilo A:  lee valor = 5
Hilo B:  lee valor = 5   ← también leyó 5, no 6
Hilo A:  escribe valor = 6
Hilo B:  escribe valor = 6  ← ¡debería ser 7!
```

Este tipo de error —donde una escritura se pierde silenciosamente— se denomina ***lost update***. Las *race conditions* pueden aparecer:

- **En el cliente:** cuando el usuario (o el código) envía requests duplicadas o simultáneas que afectan el mismo recurso.
- **En el servidor:** cuando múltiples hilos atienden requests en paralelo y comparten estado mutable sin protección.

Las secciones siguientes exploran ambos escenarios.


---

## *Race Conditions* desde el cliente

Cuando el cliente envía múltiples requests concurrentes que **modifican el mismo recurso en el servidor**, puede provocar (o verse afectado por) una *race condition*. Algunos escenarios típicos:

### Escenario 1: doble submit de formulario

El usuario hace clic dos veces rápido en "Comprar". Se envían dos requests `POST` casi simultáneas. Si el servidor no tiene protección, el pedido puede duplicarse.

**Mitigación en el cliente:** deshabilitar el botón/envío hasta recibir respuesta, o usar un token de idempotencia único por operación.

```python
import uuid

token_idempotencia = str(uuid.uuid4())  # único por intento de compra

respuesta = requests.post(
    "https://api.tienda.com/compras",
    json={"producto_id": 42, "cantidad": 1},
    headers={"Idempotency-Key": token_idempotencia}
)
```


### Escenario 2: *rate limiting* (código 429)

Enviar demasiadas requests en poco tiempo puede causar que el servidor rechace las siguientes con el código **`429 Too Many Requests`**. Esto no es una *race condition* en sí misma, pero es una consecuencia directa del uso concurrente de la API.

**Mitigación:** respetar los límites de la API, agregar retardo entre intentos (*backoff* exponencial).


### Implementación: backoff con reintentos

La función `obtener_producto_con_backoff` reintenta la solicitud usando **backoff exponencial**: si recibe un `429`, espera $2^{\text{intento}}$ segundos antes de volver a intentar ($1s$, $2s$, $4s$, …).


In [7]:
def obtener_producto_con_backoff(producto_id: int, max_reintentos: int = 3) -> dict:
    """
    Obtiene un producto y reintenta con espera exponencial si recibe 429 (Too Many Requests).
    """
    for intento in range(max_reintentos):
        respuesta = requests.get(f"{BASE}products/{producto_id}")

        if respuesta.status_code == 429:
            # Backoff exponencial: intento=0 → 1s, intento=1 → 2s, intento=2 → 4s
            espera = 2 ** intento
            print(f"  [producto {producto_id}] 429 recibido, reintentando en {espera}s...")
            time.sleep(espera)
            continue

        # Status distinto a 429: devolvemos el resultado (200, 404, etc.)
        return {
            "id": producto_id,
            "status": respuesta.status_code,
            "titulo": respuesta.json().get("title"),
        }

    # Agotados todos los reintentos sin éxito
    return {"id": producto_id, "status": 429, "titulo": "Sin respuesta tras reintentos"}


#### Simulación con `unittest.mock`

Como _DummyJSON_ no devuelve `429` fácilmente, usamos `unittest.mock.patch` para interceptar `requests.get` durante la prueba. Los productos **2 y 4** recibirán un `429` en su primer intento, y el producto **2** también en el segundo, forzando varios reintentos.


In [8]:
from unittest.mock import patch, MagicMock

_llamadas = {}  # cuenta cuántas veces se llamó por producto_id

def mock_get(url, **kwargs):
    """Simula que los productos 2 y 4 devuelven 429 en su primer intento."""
    producto_id = int(url.split("/")[-1])
    _llamadas[producto_id] = _llamadas.get(producto_id, 0) + 1

    # Productos 2 y 4: 429 en el primer intento
    if producto_id in (2, 4) and _llamadas[producto_id] == 1:
        resp = MagicMock()
        resp.status_code = 429
        return resp

    # Producto 2 también falla en el segundo intento
    if producto_id == 2 and _llamadas[producto_id] == 2:
        resp = MagicMock()
        resp.status_code = 429
        return resp

    # Resto: respuesta 200 exitosa simulada
    resp = MagicMock()
    resp.status_code = 200
    resp.json.return_value = {"title": f"Producto simulado #{producto_id}"}
    return resp


print("=== Demostración con 429 simulado ===\n")
with patch("requests.get", side_effect=mock_get):
    with ThreadPoolExecutor(max_workers=5) as executor:
        futuros = [executor.submit(obtener_producto_con_backoff, i) for i in range(1, 6)]
        resultados = [f.result() for f in as_completed(futuros)]

print()
for r in sorted(resultados, key=lambda x: x["id"]):
    print(f"  Producto {r['id']} [{r['status']}]: {r['titulo']}")


=== Demostración con 429 simulado ===

  [producto 2] 429 recibido, reintentando en 1s...
  [producto 4] 429 recibido, reintentando en 1s...
  [producto 2] 429 recibido, reintentando en 2s...

  Producto 1 [200]: Producto simulado #1
  Producto 2 [200]: Producto simulado #2
  Producto 3 [200]: Producto simulado #3
  Producto 4 [200]: Producto simulado #4
  Producto 5 [200]: Producto simulado #5


---

## *Race Conditions* en servidores web

Flask puede atender **múltiples solicitudes en paralelo** (una por hilo con `threaded=True`). Si varios hilos leen y escriben el mismo dato sin coordinación, el resultado puede corromperse tal como se describió arriba.

### El problema: estado compartido

A continuación simulamos este problema con un **servidor Flask real**: enviamos 100 solicitudes `POST` concurrentes que incrementan un contador global. Sin protección, muchas escrituras se perderán.


In [2]:
import threading
import time
import logging
import requests
from werkzeug.serving import make_server
from flask import Flask
from concurrent.futures import ThreadPoolExecutor, as_completed

logging.getLogger("werkzeug").setLevel(logging.ERROR)

# --- Servidor Flask SIN lock ---
# threaded=True: cada request es atendida en su propio hilo → race condition real
app_sin_lock = Flask("contador_sin_lock")
contador_sin_lock = 0

@app_sin_lock.route("/incrementar", methods=["POST"])
def incrementar_sin_lock():
    global contador_sin_lock
    valor = contador_sin_lock       # leer
    time.sleep(0.001)               # simula trabajo → amplifica la race condition
    contador_sin_lock = valor + 1   # escribir
    return {"ok": True}

@app_sin_lock.route("/contador", methods=["GET"])
def get_contador_sin_lock():
    return {"contador": contador_sin_lock}

srv_sin_lock = make_server("localhost", 4445, app_sin_lock, threaded=True)
threading.Thread(target=srv_sin_lock.serve_forever, daemon=True).start()

# Enviar 100 requests concurrentes al endpoint /incrementar
N = 100
with ThreadPoolExecutor(max_workers=N) as executor:
    futuros = [executor.submit(requests.post, "http://localhost:4445/incrementar")
               for _ in range(N)]
    for f in as_completed(futuros):
        f.result()

resultado = requests.get("http://localhost:4445/contador").json()["contador"]
srv_sin_lock.shutdown()

print(f"Resultado esperado : {N}")
print(f"Resultado obtenido : {resultado}")
print(f"Valores perdidos   : {N - resultado}  ← race condition!")

Resultado esperado : 100
Resultado obtenido : 56
Valores perdidos   : 44  ← race condition!


### Solución: usar un `Lock`

Un **`threading.Lock`** es un mecanismo de sincronización que garantiza que **solo un hilo a la vez** pueda ejecutar la sección crítica del código (la parte que accede al recurso compartido). Los demás hilos deben esperar hasta que el *lock* sea liberado.

```
Hilo A (request 1):  adquiere lock → lee 5 → escribe 6 → libera lock
Hilo B (request 2):  espera...     →        →            → adquiere lock → lee 6 → escribe 7 → libera lock
```

Ahora el resultado siempre es correcto.

En Flask, el patrón es:
```python
from flask import Flask
import threading

app = Flask(__name__)
lock = threading.Lock()
contador = 0

@app.route("/visitas", methods=["GET"])
def visitas():
    global contador
    with lock:          # solo un hilo a la vez entra aquí
        contador += 1
        return {"visitas": contador}
```

A continuación repetimos el mismo experimento con el `Lock` activado:


In [10]:
# --- Servidor Flask CON lock ---
app_con_lock = Flask("contador_con_lock")
_lock = threading.Lock()
contador_con_lock = 0

@app_con_lock.route("/incrementar", methods=["POST"])
def incrementar_con_lock():
    global contador_con_lock
    with _lock:                         # sección crítica protegida
        valor = contador_con_lock
        time.sleep(0.001)
        contador_con_lock = valor + 1
    return {"ok": True}

@app_con_lock.route("/contador", methods=["GET"])
def get_contador_con_lock():
    return {"contador": contador_con_lock}

srv_con_lock = make_server("localhost", 4445, app_con_lock, threaded=True)
threading.Thread(target=srv_con_lock.serve_forever, daemon=True).start()

# Mismo experimento: 100 requests concurrentes
N = 100
with ThreadPoolExecutor(max_workers=N) as executor:
    futuros = [executor.submit(requests.post, "http://localhost:4445/incrementar")
               for _ in range(N)]
    for f in as_completed(futuros):
        f.result()

resultado = requests.get("http://localhost:4445/contador").json()["contador"]
srv_con_lock.shutdown()

print(f"Resultado esperado : {N}")
print(f"Resultado obtenido : {resultado}")
print(f"Valores perdidos   : {N - resultado}  ← ninguno!")

Resultado esperado : 100
Resultado obtenido : 100
Valores perdidos   : 0  ← ninguno!
